Step 1: Import Required Libraries

Objective:
To import all the necessary libraries for dataset loading, text preprocessing, model building, training, and evaluation.

In [31]:
# Import Regular Expression library for text cleaning
import re

# Import NumPy for numerical operations
import numpy as np

# Import the AG News dataset from Hugging Face
from datasets import load_dataset

# Import Tokenizer to convert text into integer sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Import pad_sequences to make all sequences the same length
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Import to_categorical for one-hot encoding the labels
from tensorflow.keras.utils import to_categorical

# Import Sequential model to build the RNN layer by layer
from tensorflow.keras.models import Sequential

# Import the layers required for the RNN model
# Embedding: Converts words into dense vectors
# SimpleRNN: Processes sequential text data
# Dense: Output layer for classification
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

Step 2: Load the Dataset

Objective:
To load the AG News dataset and separate the training and testing text and labels.

In [32]:
# Step 2: Load Dataset


# Load the AG News dataset from Hugging Face
dataset = load_dataset("aRWA787/ag_news_dataset")

# Display the dataset information (train/test splits, number of samples)
print(dataset)

# Display the names of all columns in the training dataset
print(dataset["train"].column_names)

# Extract the news article text from the training dataset
train_texts = dataset["train"]["text"]

# Extract the corresponding class labels from the training dataset
train_labels = dataset["train"]["label"]

# Extract the news article text from the testing dataset
test_texts = dataset["test"]["text"]

# Extract the corresponding class labels from the testing dataset
test_labels = dataset["test"]["label"]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_name', 'length'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'label_name', 'length'],
        num_rows: 7600
    })
})
['text', 'label', 'label_name', 'length']


Step 3: Clean and Normalize the Text

Objective:
To preprocess the text by converting it to lowercase, removing special characters, numbers, and extra spaces.

In [33]:
# ==========================================
# Step 3: Clean and Normalize Text
# ==========================================

# Function to clean and normalize the text
def clean_text(text):

    # Convert all characters to lowercase
    text = text.lower()

    # Remove numbers, punctuation, and special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove extra spaces and trim leading/trailing spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Return the cleaned text
    return text

# Apply the cleaning function to all training text samples
train_texts = [clean_text(text) for text in train_texts]

# Apply the cleaning function to all testing text samples
test_texts = [clean_text(text) for text in test_texts]

Step 4: Tokenize the Text

Objective:
To convert the cleaned text into sequences of integer values using the Keras Tokenizer.

In [34]:


# Set the maximum vocabulary size (top 10,000 most frequent words)
vocab_size = 10000

# Create a Tokenizer object
# Words not present in the vocabulary will be replaced with <OOV>
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")

# Build the vocabulary by fitting the tokenizer on the training text
tokenizer.fit_on_texts(train_texts)

# Convert the training text into sequences of integer IDs
X_train = tokenizer.texts_to_sequences(train_texts)

# Convert the testing text into sequences of integer IDs
X_test = tokenizer.texts_to_sequences(test_texts)

Step 5: Pad the Sequences

Objective:
To make all text sequences the same length by padding shorter sequences and truncating longer ones

In [35]:


# Set the maximum sequence length to 50 words
max_length = 50

# Pad or truncate the training sequences
X_train = pad_sequences(
    X_train,
    maxlen=max_length,      # Maximum length of each sequence
    padding="post",         # Add zeros at the end of shorter sequences
    truncating="post"       # Remove extra words from the end of longer sequences
)

# Pad or truncate the testing sequences
X_test = pad_sequences(
    X_test,
    maxlen=max_length,      # Maximum length of each sequence
    padding="post",         # Add zeros at the end of shorter sequences
    truncating="post"       # Remove extra words from the end of longer sequences
)

Step 6: One-Hot Encode the Labels

Objective:
To convert the class labels into one-hot encoded vectors for multi-class classification.

In [36]:

# Step 6: One-Hot Encode Labels


# Find the total number of unique classes in the dataset
num_classes = len(set(train_labels))

# Convert the training labels into one-hot encoded vectors
# Example: Class 2 → [0, 0, 1, 0]
y_train = to_categorical(train_labels, num_classes=num_classes)

# Convert the testing labels into one-hot encoded vectors
y_test = to_categorical(test_labels, num_classes=num_classes)

Step 7: Build the Simple RNN Model

Objective:
To create a Sequential RNN model with an Embedding layer, a SimpleRNN layer, and a Dense output layer for classifying news articles

In [37]:

# Create a Sequential model
model = Sequential()

# Add an Embedding layer
# Converts integer word IDs into dense vector representations
model.add(
    Embedding(
        input_dim=vocab_size,      # Size of the vocabulary
        output_dim=64,             # Dimension of each word embedding
        input_length=max_length    # Length of each input sequence
    )
)

# Add a Simple RNN layer with 64 neurons
# Learns sequential patterns and dependencies in the text
model.add(SimpleRNN(64))

# Add the output layer
# Softmax activation predicts the probability of each news category
model.add(Dense(num_classes, activation="softmax"))

Step 8: Compile the Model

Objective:
To configure the RNN model using the Adam optimizer, categorical cross-entropy loss function, and accuracy metric

In [38]:

# Step 8: Compile the RNN Model


# Compile the model by specifying the optimizer,
# loss function, and evaluation metric
model.compile(
    optimizer="adam",                  # Optimizer used to update model weights
    loss="categorical_crossentropy",   # Loss function for multi-class classification
    metrics=["accuracy"]               # Evaluate model performance using accuracy
)

# Display the architecture and summary of the model
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Step 9: Train the Model

Objective:
To train the RNN model using the training data for multiple epochs and validate its performance during training

In [39]:


# Train the model using the training data
history = model.fit(
    X_train,                   # Input training sequences
    y_train,                   # One-hot encoded training labels
    epochs=5,                  # Number of times the model processes the entire training dataset
    batch_size=64,             # Number of samples processed before updating the model weights
    validation_split=0.2       # Use 20% of the training data for validation
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 36s 22ms/step - accuracy: 0.8099 - loss: 0.5414 - val_accuracy: 0.8654 - val_loss: 0.4026
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 35s 23ms/step - accuracy: 0.8983 - loss: 0.3368 - val_accuracy: 0.8773 - val_loss: 0.3944
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 34s 23ms/step - accuracy: 0.9111 - loss: 0.2925 - val_accuracy: 0.8641 - val_loss: 0.4249
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 34s 22ms/step - accuracy: 0.9223 - loss: 0.2507 - val_accuracy: 0.8801 - val_loss: 0.3743
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 40s 22ms/step - accuracy: 0.9330 - loss: 0.2143 - val_accuracy: 0.8713 - val_loss: 0.4435


Step 10: Evaluate the Model

Objective:
To evaluate the trained RNN model on the test dataset and calculate the test loss and accuracy.

In [40]:


# Evaluate the trained RNN model on the test dataset
loss, accuracy = model.evaluate(X_test, y_test)

# Print the loss value obtained on the test dataset
print("\nTest Loss :", loss)

# Print the accuracy achieved on the test dataset
print("Test Accuracy :", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8834 - loss: 0.4067

Test Loss : 0.40672802925109863
Test Accuracy : 0.8834210634231567


Step 11: Predict a New News Article

Objective:
To classify a new news article by preprocessing the input, converting it into a padded sequence, and predicting its category using the trained RNN model.

In [41]:

# Create a sample news article for prediction
sample = ["Apple launches a new AI-powered smartphone."]

# Clean and normalize the sample text
sample = [clean_text(text) for text in sample]

# Convert the sample text into a sequence of integer IDs
sample_seq = tokenizer.texts_to_sequences(sample)

# Pad the sequence to match the input length used during training
sample_pad = pad_sequences(
    sample_seq,
    maxlen=max_length,
    padding="post"
)

# Predict the class probabilities for the sample
prediction = model.predict(sample_pad)

# Find the class with the highest probability
predicted_class = np.argmax(prediction)

# Display the predicted class label
print("Predicted Class:", predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
Predicted Class: 3
